<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/dentistry/lecture_9/%D0%9F%D1%80%D0%B0%D0%BA%D1%82%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B0%D1%8F_%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%B0_%E2%84%96_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Практическая работа № 9: Retrieval-Augmented Generation (RAG) в стоматологии и смежных медицинских областях

## Введение

В лекции №9 мы познакомились с технологией **Retrieval-Augmented Generation (RAG)**, которая позволяет LLM отвечать точно, опираясь на внешнюю базу знаний. Мы разобрали:

- проблему галлюцинаций LLM и почему она критична для медицины (и особенно стоматологии, где точность диагностики и лечения напрямую влияет на здоровье пациента),
- архитектуру RAG: поиск (retrieval), подстановка (augmentation), генерация (generation),
- чанкинг, эмбеддинги и векторное хранилище Chroma,
- как запустить RAG-систему в Google Colab с локальной LLM через Ollama,
- примеры применения RAG в стоматологии (подготовка к приёму, научный поиск, клинические протоколы, обучение),
- этические аспекты использования RAG.

Теперь вам предстоит применить эти знания на практике.

**Цель работы** — закрепить навыки:
- установки и настройки Ollama в Google Colab,
- создания базы знаний из стоматологических текстов,
- построения RAG-пайплайна (чанкинг, эмбеддинги, векторное хранилище, генерация),
- тестирования RAG-системы на реальных вопросах,
- сравнения качества ответов разных моделей,
- критической оценки результатов и этических аспектов.

**Междисциплинарная перспектива:** хотя основное внимание уделяется стоматологии, все задания и принципы могут быть легко адаптированы для других медицинских специальностей. Неонатолог может использовать RAG для поиска протоколов выхаживания недоношенных, рентгенолог — для работы с классификациями (BI-RADS, PI-RADS), терапевт — для доступа к клиническим рекомендациям, микробиолог — для интерпретации антибиотикограмм. Освоив базовые навыки на стоматологических примерах, вы сможете применять их в своей узкой области.

---

## Подготовка рабочей среды

Для выполнения работы вам понадобится **Google Colab** — бесплатная облачная среда с доступом к GPU. Вам не нужно устанавливать ПО на свой компьютер; достаточно браузера и интернета.

**Минимальные требования:**
- Аккаунт Google (для доступа к Colab).
- Браузер (Chrome, Firefox, Safari, Edge).
- Стабильное интернет-соединение (для скачивания моделей).

**Что вы будете делать:**
1. Создадите новый ноутбук в Colab.
2. Установите Ollama и скачаете лёгкую модель (`llama3.2:1b` или `qwen2.5:3b`).
3. Установите библиотеки (chromadb, sentence-transformers, langchain).
4. Создадите базу знаний из стоматологических текстов (клинические признаки, протоколы лечения, классификации).
5. Напишете функцию RAG и протестируете её на вопросах.
6. Сравните ответы разных моделей.
7. Проведёте этический анализ.

---

## Часть 1. Теоретические вопросы (для самопроверки)

Перед выполнением практических заданий письменно ответьте на следующие вопросы. Это поможет убедиться, что вы понимаете ключевые концепции.

1. Что такое галлюцинации LLM? Почему они возникают и чем опасны в стоматологии (и в медицине в целом)?

2. Что такое RAG (Retrieval-Augmented Generation)? Опишите три основных шага RAG-пайплайна.

3. В чём разница между обычным запросом к LLM и запросом с RAG?

4. Что такое чанкинг? Почему он важен в RAG? Какие стратегии чанкинга вы знаете? Какая стратегия лучше подходит для стоматологических учебников?

5. Что такое эмбеддинги? Как они помогают в поиске релевантных фрагментов? Приведите пример: как эмбеддинги могут отличить текст о пульпите от текста о гигиене полости рта.

6. Что такое векторное хранилище? Назовите пример такого хранилища (из лекции – Chroma) и объясните, как оно работает.

7. Как установить Ollama в Google Colab? Какие команды для этого нужны? Почему в Colab рекомендуется использовать лёгкую модель (например, `llama3.2:1b`)?

8. Какую модель эмбеддингов рекомендовано использовать для русского языка в RAG? Почему?

9. Какие этические требования нужно соблюдать при использовании RAG в клинической стоматологии? Назовите не менее трёх.

10. **Рефлексивный вопрос:** как вы видите использование RAG в своей будущей практике (стоматология или смежная специальность)? Какие источники вы бы добавили в базу знаний?

---

## Часть 2. Практические задания

Все задания выполняйте в Google Colab в одном ноутбуке. Код должен быть снабжён комментариями на русском языке. В конце каждого задания приводите краткий анализ результатов.

---

### Задание 1. Установка и запуск Ollama в Colab

**Описание.** В этом задании вы установите Ollama в Google Colab, запустите сервер и скачаете лёгкую модель для тестирования.

**Требуется:**

1. Создайте новый ноутбук в Google Colab и переименуйте его в `RAG_Dental`.

2. Выполните код для очистки старых файлов, установки Ollama и запуска сервера (код из лекции, раздел 6.2.2, адаптированный под стоматологию – просто замените названия).

3. Скачайте модель **`llama3.2:1b`** (или `qwen2.5:3b`, если позволяет память Colab).

4. Проверьте, что модель работает, отправив тестовый запрос через `ollama run`: например, "Привет, как ты можешь помочь стоматологу?"

5. Выведите список установленных моделей.

**Что сдать:** скриншоты выполнения всех шагов (терминал с командами) и краткое описание (1 страница).

```python
# Ваш код решения задачи (вставьте сюда код из лекции, адаптированный под стоматологию):
```

---

### Задание 2. Создание базы знаний из стоматологических текстов

**Описание.** В этом задании вы создадите базу знаний из стоматологических текстов. Вы используете модель эмбеддингов `multilingual-e5-large`, разобьёте тексты на чанки и сохраните их в векторном хранилище Chroma.

**Требуется:**

1. Установите необходимые библиотеки (`chromadb`, `sentence-transformers`, `langchain-text-splitters`).

2. Загрузите эмбеддинг-модель `intfloat/multilingual-e5-large`.

3. Подготовьте массив из 5–7 стоматологических текстов. Примеры (можно использовать свои):
   - "Острый пульпит характеризуется самопроизвольной болью, усиливающейся ночью, реакцией на холодное и горячее. Перкуссия слабоположительная. Лечение: эндодонтическое."
   - "Хронический периодонтит: боль при накусывании, отёк десны, возможно образование свища. На рентгенограмме очаг разрежения костной ткани. Лечение: эндодонтическое или хирургическое."
   - "Гингивит: кровоточивость дёсен при чистке зубов, отёк десневого края, неприятный запах. Лечение: профессиональная гигиена, обучение уходу."
   - "Кариес: начальный кариес проявляется белым пятном, при прогрессировании образуется полость. Лечение: удаление поражённых тканей и пломбирование."
   - "Профессиональная гигиена полости рта включает снятие зубных отложений ультразвуком и полировку. Рекомендуется каждые 6 месяцев."
   - "Пародонтит: воспаление тканей пародонта, потеря прикрепления, подвижность зубов. Лечение: пародонтологическое (скейлинг, кюретаж), при необходимости хирургическое."

4. Разбейте тексты на чанки размером 300 символов с перекрытием 50 символов.

5. Создайте эмбеддинги для всех чанков.

6. Создайте коллекцию в Chroma и добавьте чанки с их эмбеддингами.

7. Выведите количество созданных чанков и проверьте, что база данных сохранена (например, через `collection.count()`).

**Что сдать:** код ячейки, скриншот результатов (количество чанков, подтверждение сохранения) и краткий анализ (1 страница).

```python
# Ваш код решения задачи:
```

---

### Задание 3. Реализация RAG-функции

**Описание.** Напишите функцию `rag_query(query, top_k=3)`, которая выполняет полный RAG-пайплайн:
1. Превращает вопрос в эмбеддинг.
2. Ищет в Chroma top_k самых похожих чанков.
3. Формирует промпт с найденными фрагментами.
4. Отправляет запрос к Ollama (модель `llama3.2:1b`) и возвращает ответ.

**Требуется:**

1. Реализуйте функцию с явными шагами и выводом промежуточных результатов (как в лекции, но с стоматологическим промптом, например: "Ты – стоматологический ассистент...").

2. Используйте температуру 0.3 для детерминированных ответов.

3. Добавьте обработку ошибок (если сервер не отвечает, если модель не найдена).

4. Протестируйте функцию на вопросе: *«Каковы симптомы острого пульпита?»*.

5. Выведите найденные фрагменты и ответ модели.

**Что сдать:** код функции, результат её работы на тестовом вопросе, анализ (насколько точен ответ, есть ли ссылки на источники).

```python
# Ваш код решения задачи:
```

---

### Задание 4. Тестирование RAG на разных вопросах

**Описание.** Протестируйте RAG-систему на 3–4 вопросах разного типа:
- Вопрос, на который есть точный ответ в базе (например, о симптомах пульпита).
- Вопрос, на который есть частичный ответ (например, о лечении периодонтита).
- Вопрос, на который нет ответа (например, о имплантации, если её нет в базе).
- Вопрос, требующий синтеза информации из нескольких фрагментов (например, "Какие заболевания вызывают кровоточивость дёсен?").

**Требуется:**

1. Для каждого вопроса запишите:
   - Сам вопрос.
   - Найденные фрагменты (с указанием степени сходства).
   - Ответ модели.
   - Вашу оценку: правильный ли ответ, есть ли галлюцинации, ссылается ли модель на источники.

2. Напишите краткий вывод: насколько хорошо работает RAG-система? Какие вопросы она отвечает лучше, какие хуже? Как это может быть использовано в клинической практике?

**Что сдать:** таблица с вопросами, найденными фрагментами, ответами и вашей оценкой (можно в формате Markdown или в текстовой ячейке).

```python
# Ваш код для тестирования (вызов rag_query для каждого вопроса):
```

---

### Задание 5. Сравнение двух моделей в RAG

**Описание.** Сравните качество ответов двух разных моделей в RAG-системе. Используйте `llama3.2:1b` и `qwen2.5:3b` (или `llama3.2:3b`). Для этого скачайте вторую модель и адаптируйте функцию, чтобы можно было передавать имя модели.

**Требуется:**

1. Скачайте вторую модель (`ollama pull qwen2.5:3b` или `llama3.2:3b`).

2. Напишите функцию `rag_query_with_model(query, model_name, top_k=3)`, которая принимает имя модели.

3. Для одного и того же вопроса (например, *«Какие признаки гингивита?»*) получите ответы от обеих моделей.

4. Сравните:
   - Время ответа (засеките с помощью `time.time()`).
   - Качество ответа (полнота, точность, ссылки на источники).
   - Стиль ответа (формальный/неформальный, эмпатичный/нейтральный).

5. Напишите вывод: какая модель лучше подходит для RAG в стоматологии и почему? Учитывайте ограничения памяти и скорости в Colab.

**Что сдать:** код сравнения, таблица с результатами, рефлексия (1–2 страницы).

```python
# Ваш код решения задачи:
```

---

### Задание 6. Расширение базы знаний (творческое задание)

**Описание.** Добавьте в базу знаний свои тексты по стоматологии (например, из учебников, статей, клинических рекомендаций). Можно взять тексты из интернета (в рамках добросовестного использования) или использовать собственные конспекты. После добавления протестируйте систему на новых вопросах.

**Требуется:**

1. Выберите тему (например, «эндодонтия», «пародонтология», «ортопедия», «хирургическая стоматология»).

2. Добавьте минимум 3 новых текста в массив `documents` и пересоздайте базу знаний (или добавьте их в существующую коллекцию через `collection.add`).

3. Сформулируйте 3 вопроса по новой теме, на которые должны быть ответы в добавленных текстах.

4. Запустите RAG и запишите ответы.

5. Напишите рефлексию (1 страница): насколько легко было добавлять новые тексты? Как изменилось качество ответов? Что можно улучшить в системе (например, использовать метаданные, фильтры)?

**Что сдать:** список добавленных текстов (или ссылки на них), вопросы и ответы, рефлексия.

```python
# Ваш код для добавления новых текстов и тестирования:
```

---

### Задание 7. Этический анализ (обязательное)

**Описание.** Представьте, что вы — руководитель стоматологической клиники (или медицинского центра). Вы хотите внедрить RAG-систему для поддержки работы врачей: она будет отвечать на вопросы по диагностическим критериям, клиническим протоколам, научной литературе и помогать в подготовке к приёму.

**Напишите этический протокол** (2–3 страницы), в котором осветите:

1. **Информированное согласие** — что пациент должен знать об использовании RAG-системы? Какие пункты обязательно включить в форму согласия (например, что ответы не являются диагнозом, что решение принимает врач)?

2. **Источники информации** — как вы будете отбирать тексты для базы знаний? Какие критерии качества источников (рецензируемые учебники, официальные клинические рекомендации, избегать непроверенных блогов)? Как часто обновлять базу?

3. **Конфиденциальность и безопасность** — где хранятся данные (локально или в облаке)? Кто имеет доступ? Как обеспечивается анонимизация данных пациента при формулировании вопросов? (упомяните 152-ФЗ и врачебную тайну).

4. **Человеческий контроль** — как организовать проверку ответов системы врачом? Кто отвечает за финальные решения? (врач всегда подтверждает).

5. **Обработка ошибок** — что делать, если система дала неверный ответ (галлюцинация)? Как это повлияет на клиническое решение? Опишите процедуру сообщения об ошибках и обновления базы.

6. **Прозрачность** — как вы будете показывать пациенту и коллегам, что ответ сгенерирован ИИ и на каких источниках он основан? (например, в интерфейсе отображать источники).

**Что сдать:** этический протокол в формате Word/PDF (или текстовой ячейкой в Notebook).

```text
# Ваш анализ
```

---

## Часть 3. Комплексное задание (повышенной сложности) — по желанию

**Задание 8. Создание веб-интерфейса для RAG с помощью Streamlit**

**Описание.** Разработайте простое веб-приложение на Streamlit, которое позволяет загружать тексты в базу знаний и задавать вопросы через RAG. Приложение может быть запущено локально или в Colab (через `ngrok` для публичного доступа).

**Требуется:**

1. Установите Streamlit: `!pip install streamlit` (в Colab можно использовать `streamlit run` через `pyngrok`).

2. Создайте приложение с:
   - Полем для загрузки текстовых файлов (или ввода текста).
   - Кнопкой «Добавить в базу знаний».
   - Полем для ввода вопроса.
   - Кнопкой «Получить ответ».
   - Отображением ответа и источников (найденных фрагментов).

3. Запустите приложение и продемонстрируйте его работу (скриншоты).

**Что сдать:** код приложения, скриншоты работы, краткое описание.

```python
# Ваш код для Streamlit-приложения:
```

## Критерии оценки

| Компонент | Процент | Описание |
|-----------|---------|----------|
| Теоретические вопросы (Часть 1) | 15% | Полнота и правильность ответов |
| Задание 1 (установка Ollama) | 10% | Корректность установки, скриншоты |
| Задание 2 (база знаний) | 10% | Корректность создания базы, чанкинг, эмбеддинги |
| Задание 3 (RAG-функция) | 15% | Работающая функция, вывод промежуточных результатов |
| Задание 4 (тестирование) | 10% | Разнообразие вопросов, анализ ответов |
| Задание 5 (сравнение моделей) | 10% | Качество сравнения, выводы |
| Задание 6 (расширение базы) | 10% | Добавление текстов, тестирование, рефлексия |
| Задание 7 (этический протокол) | 20% | Полнота, аргументированность, практичность |
| Задание 8 (Streamlit, бонус) | +5% | Работающее приложение, демонстрация |

---

## Требования к сдаче

- Пришлите **один файл** (Jupyter Notebook `.ipynb`) со всеми заданиями, кодом и текстовыми комментариями.
- Этический протокол (Задание 7) приложите отдельным файлом (`.pdf` или `.docx`) или в виде текстовой ячейки в Notebook.
- Скриншоты вставьте в Notebook или приложите отдельными файлами.
- Убедитесь, что код выполняется без ошибок. Укажите версии библиотек (в Colab они будут актуальны).
- Все результаты анализа должны сопровождаться интерпретацией с точки зрения врача-стоматолога (или вашей специальности).

---

## Заключение

Данная практическая работа проведёт вас через полный цикл создания RAG-системы для стоматологии — от установки Ollama до сравнения моделей и этического анализа. Вы не только освоите технические навыки, но и научитесь критически оценивать качество и безопасность ИИ-инструментов в клинической практике.

**Главный вывод:** RAG — это мощный инструмент, который превращает LLM из «гадалки» в «исследователя-ассистента», но его использование требует строгого контроля источников, человеческой проверки и соблюдения этических норм. Ответственность за точность и безопасность всегда остаётся за врачом.

**Междисциплинарное применение:** освоенные навыки могут быть перенесены на любую медицинскую область. Замените базу знаний на неонатальные протоколы, рентгенологические классификации или микробиологические справочники — и вы получите персонального ассистента для своей специальности.

---

**Срок выполнения: 2 недели.**